EDA: Transformacion Agricola y Perdida Forestal en Mexico (2001-2023)
Fuentes: Global Forest Watch (MEX.xlsx) + SIAP (DGSIAP/)


In [3]:
import os
import glob
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")

---- CONFIGURACION ----

In [4]:
DATA_DIR = "Data"
DGSIAP_DIR = os.path.join(DATA_DIR, "DGSIAP")
GFW_PATH = os.path.join(DATA_DIR, "MEX.xlsx")
GFW_THRESHOLD = 30  # umbral estandar para bosques tropicales
OUTPUT_DIR = "eda_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PALETTE = {
    "bosque": "#2E7D32",
    "agricola": "#E65100",
    "neutro": "#1565C0",
    "alerta": "#B71C1C",
    "gris": "#546E7A",
}

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "grid.linestyle": "--",
    }
)

---- FUNCIONES DE CARGA ----

In [5]:

def cargar_archivo_disfrazado(ruta):
    tablas = pd.read_html(ruta, encoding="utf-8")
    return tablas[0]

def limpiar_siap(df):
    """Aplana el multiindex, renombra columnas y elimina la fila Total."""
    df.columns = ["num", "entidad_o_cultivo", "sembrada", "cosechada", "valor"]
    df = df[df["entidad_o_cultivo"].notna()]
    df = df[df["entidad_o_cultivo"].str.strip().str.lower() != "total"]
    df = df[df["entidad_o_cultivo"].str.strip() != "Entidad"]
    df = df[df["entidad_o_cultivo"].str.strip() != "Cultivo"]
    for col in ["sembrada", "cosechada", "valor"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["sembrada"])
    df = df.reset_index(drop=True)
    return df


def cargar_todos_los_temporales(carpeta, tipo):
    patron = os.path.join(carpeta, f"*_{tipo}.xls")
    archivos = sorted(glob.glob(patron))
    if not archivos:
        print(f"  AVISO: No se encontraron archivos para '{tipo}' en {carpeta}")
        return pd.DataFrame()
    dfs = []
    for ruta in archivos:
        year = int(os.path.basename(ruta)[:4])
        df = limpiar_siap(cargar_archivo_disfrazado(ruta))
        df["year"] = year
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def cargar_gfw(ruta, sheet, threshold=30):
    df = pd.read_excel(ruta, sheet_name=sheet)
    if "threshold" in df.columns:
        df = df[df["threshold"] == threshold].copy()
    return df

---- CARGA DE DATOS ----

In [ ]:
print("=" * 60)
print("CARGA DE DATOS")
print("=" * 60)

print("\n[1/4] Cargando archivos SIAP - estados...")
df_estados = cargar_todos_los_temporales(DGSIAP_DIR, "estados")
df_estados = df_estados.rename(columns={"entidad_o_cultivo": "estado"})

print("[2/4] Cargando archivos SIAP - cultivos...")
df_cultivos = cargar_todos_los_temporales(DGSIAP_DIR, "cultivo")
df_cultivos = df_cultivos.rename(columns={"entidad_o_cultivo": "cultivo"})

print("[3/4] Cargando GFW - perdida forestal por estado...")
gfw_estados = cargar_gfw(GFW_PATH, "Subnational 1 tree cover loss", GFW_THRESHOLD)

print("[4/4] Cargando GFW - perdida forestal nacional...")
gfw_nacional = cargar_gfw(GFW_PATH, "Country tree cover loss", GFW_THRESHOLD)

years_siap = sorted(df_estados["year"].unique()) if not df_estados.empty else []
print(f"\nYears disponibles SIAP: {years_siap}")
print(f"Estados SIAP: {df_estados['estado'].nunique() if not df_estados.empty else 0}")
print(
    f"Cultivos unicos: {df_cultivos['cultivo'].nunique() if not df_cultivos.empty else 0}"
)
print(f"Estados GFW: {gfw_estados['subnational1'].nunique()}")

CARGA DE DATOS

[1/4] Cargando archivos SIAP - estados...
[2/4] Cargando archivos SIAP - cultivos...
[3/4] Cargando GFW - perdida forestal por estado...
[4/4] Cargando GFW - perdida forestal nacional...

Anios disponibles SIAP: [np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Estados SIAP: 32
Cultivos unicos: 322
Estados GFW: 32


---- SECCION 1: VISION GENERAL DE LOS DATASETS ----

In [8]:
print("\n" + "=" * 60)
print("SECCION 1: VISION GENERAL DE LOS DATASETS")
print("=" * 60)

columnas_loss = [c for c in gfw_estados.columns if c.startswith("tc_loss_ha_")]
years_gfw = [int(c.replace("tc_loss_ha_", "")) for c in columnas_loss]

print("\n--- GFW Subnacional (threshold=30) ---")
print(f"Shape: {gfw_estados.shape}")
print(f"Columnas: {list(gfw_estados.columns)}")
print(
    f"Years de perdida disponibles: {min(years_gfw)}-{max(years_gfw)} ({len(years_gfw)} year)"
)
print(f"\nEstadisticas de cobertura arborea inicial (2000):")
print(gfw_estados[["extent_2000_ha", "gain_2000-2020_ha"]].describe().round(0))

if not df_estados.empty:
    print("\n--- SIAP Estados ---")
    print(f"Shape: {df_estados.shape}")
    print(f"Columnas: {list(df_estados.columns)}")
    print(f"\nEstadisticas basicas:")
    print(df_estados[["sembrada", "cosechada", "valor"]].describe().round(0))

    print("\n--- SIAP Cultivos ---")
    print(f"Shape: {df_cultivos.shape}")
    print(f"Tipos de cultivo unicos: {df_cultivos['cultivo'].nunique()}")

# Valores faltantes GFW
print("\n--- Valores faltantes GFW (estados) ---")
nulos_gfw = gfw_estados[columnas_loss].isna().sum()
if nulos_gfw.sum() == 0:
    print("Sin valores faltantes en columnas de perdida forestal.")
else:
    print(nulos_gfw[nulos_gfw > 0])

if not df_estados.empty:
    print("\n--- Valores faltantes SIAP estados ---")
    nulos_siap = df_estados[["sembrada", "cosechada", "valor"]].isna().sum()
    print(nulos_siap)


SECCION 1: VISION GENERAL DE LOS DATASETS

--- GFW Subnacional (threshold=30) ---
Shape: (32, 30)
Columnas: ['country', 'subnational1', 'threshold', 'area_ha', 'extent_2000_ha', 'extent_2010_ha', 'gain_2000-2020_ha', 'tc_loss_ha_2001', 'tc_loss_ha_2002', 'tc_loss_ha_2003', 'tc_loss_ha_2004', 'tc_loss_ha_2005', 'tc_loss_ha_2006', 'tc_loss_ha_2007', 'tc_loss_ha_2008', 'tc_loss_ha_2009', 'tc_loss_ha_2010', 'tc_loss_ha_2011', 'tc_loss_ha_2012', 'tc_loss_ha_2013', 'tc_loss_ha_2014', 'tc_loss_ha_2015', 'tc_loss_ha_2016', 'tc_loss_ha_2017', 'tc_loss_ha_2018', 'tc_loss_ha_2019', 'tc_loss_ha_2020', 'tc_loss_ha_2021', 'tc_loss_ha_2022', 'tc_loss_ha_2023']
Years de perdida disponibles: 2001-2023 (23 year)

Estadisticas de cobertura arborea inicial (2000):
       extent_2000_ha  gain_2000-2020_ha
count            32.0               32.0
mean        1661786.0            44472.0
std         1547051.0            46198.0
min           13535.0              408.0
25%          301611.0             8932.

---- SECCION 2: EVOLUCION FORESTAL NACIONAL ----

In [9]:
print("\n" + "=" * 60)
print("SECCION 2: PERDIDA FORESTAL NACIONAL (GFW)")
print("=" * 60)

# Serie temporal de perdida nacional
loss_nacional = gfw_nacional[columnas_loss].iloc[0]
loss_nacional.index = years_gfw
perdida_total = loss_nacional.sum()
print(
    f"\nPerdida forestal acumulada 2001-2023 (threshold={GFW_THRESHOLD}%): {perdida_total:,.0f} ha"
)
print(
    f"Year con mayor perdida: {loss_nacional.idxmax()} ({loss_nacional.max():,.0f} ha)"
)
print(
    f"Year con menor perdida: {loss_nacional.idxmin()} ({loss_nacional.min():,.0f} ha)"
)
print(f"Promedio anual: {loss_nacional.mean():,.0f} ha/year")

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle(
    "Perdida Forestal Nacional en Mexico (GFW, threshold=30%)",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

ax = axes[0]
colores_barras = [
    PALETTE["alerta"] if v > loss_nacional.mean() else PALETTE["bosque"]
    for v in loss_nacional.values
]
bars = ax.bar(
    loss_nacional.index,
    loss_nacional.values / 1000,
    color=colores_barras,
    alpha=0.85,
    edgecolor="white",
    linewidth=0.5,
)
ax.axhline(
    loss_nacional.mean() / 1000,
    color=PALETTE["gris"],
    linestyle="--",
    linewidth=1.5,
    label=f"Promedio: {loss_nacional.mean()/1000:.0f} mil ha/year",
)
ax.set_title("Perdida anual de cobertura arborea", fontsize=12)
ax.set_ylabel("Miles de hectareas perdidas")
ax.set_xlabel("")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}k"))

ax2 = axes[1]
perdida_acumulada = loss_nacional.cumsum() / 1e6
ax2.fill_between(
    perdida_acumulada.index,
    perdida_acumulada.values,
    alpha=0.3,
    color=PALETTE["alerta"],
)
ax2.plot(
    perdida_acumulada.index,
    perdida_acumulada.values,
    color=PALETTE["alerta"],
    linewidth=2.5,
)
ax2.set_title("Perdida forestal acumulada 2001-2023", fontsize=12)
ax2.set_ylabel("Millones de hectareas (acumulado)")
ax2.set_xlabel("Year")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}M ha"))

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "01_perdida_forestal_nacional.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.close()
print("\nGuardado: 01_perdida_forestal_nacional.png")


SECCION 2: PERDIDA FORESTAL NACIONAL (GFW)

Perdida forestal acumulada 2001-2023 (threshold=30%): 4,886,723 ha
Year con mayor perdida: 2019 (327,438 ha)
Year con menor perdida: 2003 (152,559 ha)
Promedio anual: 212,466 ha/year

Guardado: 01_perdida_forestal_nacional.png


---- SECCION 3: PERDIDA FORESTAL POR ESTADO ----

In [10]:
print("\n" + "=" * 60)
print("SECCION 3: PERDIDA FORESTAL POR ESTADO")
print("=" * 60)

gfw_estados["perdida_total"] = gfw_estados[columnas_loss].sum(axis=1)
gfw_estados["perdida_pct_extent"] = (
    gfw_estados["perdida_total"] / gfw_estados["extent_2000_ha"]
) * 100

top_estados = gfw_estados.sort_values("perdida_total", ascending=False).head(10)
print("\nTop 10 estados por perdida forestal absoluta (ha):")
print(
    top_estados[
        ["subnational1", "perdida_total", "perdida_pct_extent", "extent_2000_ha"]
    ].to_string(index=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
    "Perdida Forestal por Estado (2001-2023, threshold=30%)",
    fontsize=14,
    fontweight="bold",
)

ax = axes[0]
estados_plot = gfw_estados.sort_values("perdida_total", ascending=True).tail(15)
bars = ax.barh(
    estados_plot["subnational1"],
    estados_plot["perdida_total"] / 1000,
    color=PALETTE["bosque"],
    alpha=0.8,
    edgecolor="white",
)
ax.set_title("Top 15: Perdida absoluta (miles de ha)", fontsize=11)
ax.set_xlabel("Miles de hectareas perdidas")
for bar, val in zip(bars, estados_plot["perdida_total"] / 1000):
    if val > 50:
        ax.text(
            val + 5,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.0f}k",
            va="center",
            fontsize=8,
            color=PALETTE["gris"],
        )

ax2 = axes[1]
estados_pct = gfw_estados.sort_values("perdida_pct_extent", ascending=True).tail(15)
bars2 = ax2.barh(
    estados_pct["subnational1"],
    estados_pct["perdida_pct_extent"],
    color=PALETTE["alerta"],
    alpha=0.8,
    edgecolor="white",
)
ax2.set_title("Top 15: Perdida relativa (% de cobertura inicial)", fontsize=11)
ax2.set_xlabel("% de cobertura arborea del 2000 perdida")
for bar, val in zip(bars2, estados_pct["perdida_pct_extent"]):
    if val > 5:
        ax2.text(
            val + 0.2,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%",
            va="center",
            fontsize=8,
            color=PALETTE["gris"],
        )

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "02_perdida_forestal_estados.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.close()
print("\nGuardado: 02_perdida_forestal_estados.png")


SECCION 3: PERDIDA FORESTAL POR ESTADO

Top 10 estados por perdida forestal absoluta (ha):
subnational1  perdida_total  perdida_pct_extent  extent_2000_ha
    Campeche         906743           19.759076         4588995
     Chiapas         783374           16.172061         4843996
Quintana Roo         599593           15.964344         3755826
     Yucatán         522499           17.986611         2904933
    Veracruz         496448           16.666359         2978743
      Oaxaca         431916            8.550102         5051589
     Tabasco         163542           19.780666          826777
  Tamaulipas         155988            6.456874         2415844
    Guerrero         135546            4.404597         3077376
   Michoacán         103706            4.323526         2398644

Guardado: 02_perdida_forestal_estados.png


---- SECCION 4: HEATMAP FORESTAL POR ESTADO Y YEAR ----

In [11]:
print("\n" + "=" * 60)
print("SECCION 4: HEATMAP DE PERDIDA FORESTAL (ESTADO x YEAR)")
print("=" * 60)

top_n = 16
top_estados_names = (
    gfw_estados.sort_values("perdida_total", ascending=False)
    .head(top_n)["subnational1"]
    .tolist()
)
gfw_top = gfw_estados[gfw_estados["subnational1"].isin(top_estados_names)].set_index(
    "subnational1"
)
matriz_loss = gfw_top[columnas_loss].copy()
matriz_loss.columns = years_gfw
matriz_loss = matriz_loss.loc[top_estados_names]

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(
    matriz_loss / 1000,
    ax=ax,
    cmap="YlOrRd",
    linewidths=0.3,
    linecolor="white",
    annot=False,
    fmt=".0f",
    cbar_kws={"label": "Miles de ha perdidas", "shrink": 0.8},
)
ax.set_title(
    f"Perdida forestal anual por estado (Top {top_n}, threshold=30%)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "03_heatmap_forestal.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print(f"Guardado: 03_heatmap_forestal.png")


SECCION 4: HEATMAP DE PERDIDA FORESTAL (ESTADO x YEAR)
Guardado: 03_heatmap_forestal.png


---- SECCION 5: EVOLUCION AGRICOLA NACIONAL (SIAP) ----

In [12]:
if not df_estados.empty and len(years_siap) >= 2:
    print("\n" + "=" * 60)
    print("SECCION 5: EVOLUCION AGRICOLA NACIONAL (SIAP)")
    print("=" * 60)

    total_por_year = df_estados.groupby("year")["sembrada"].sum() / 1e6
    print("\nSuperficie sembrada total por year (millones de ha):")
    print(total_por_year.round(2).to_string())
    if len(years_siap) >= 2:
        cambio_total = total_por_year.iloc[-1] - total_por_year.iloc[0]
        print(
            f"\nCambio neto {years_siap[0]}-{years_siap[-1]}: {cambio_total:.2f} millones de ha ({cambio_total/total_por_year.iloc[0]*100:.1f}%)"
        )

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(
        total_por_year.index,
        total_por_year.values,
        marker="o",
        color=PALETTE["agricola"],
        linewidth=2.5,
        markersize=7,
    )
    ax.fill_between(
        total_por_year.index,
        total_por_year.values,
        alpha=0.15,
        color=PALETTE["agricola"],
    )
    ax.set_title(
        "Superficie Agricola Total Sembrada en Mexico", fontsize=13, fontweight="bold"
    )
    ax.set_ylabel("Millones de hectareas")
    ax.set_xlabel("Year")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}M ha"))
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "04_superficie_agricola_total.png"),
        dpi=150,
        bbox_inches="tight",
    )
    plt.close()
    print("Guardado: 04_superficie_agricola_total.png")


SECCION 5: EVOLUCION AGRICOLA NACIONAL (SIAP)

Superficie sembrada total por year (millones de ha):
year
2001    21.59
2002    21.64
2003    21.73
2004    21.86
2005    21.62
2006    21.42
2007    21.71
2008    21.88
2009    21.81
2010    21.93
2011    22.11
2012    21.88
2013    22.09
2014    22.18
2015    22.12
2016    21.91
2017    21.56
2018    21.14
2019    20.64
2020    18.11
2021    18.13
2022    20.54
2023    20.00

Cambio neto 2001-2023: -1.59 millones de ha (-7.4%)
Guardado: 04_superficie_agricola_total.png


---- SECCION 6: CULTIVOS CLAVE ----

In [13]:


if not df_cultivos.empty and len(years_siap) >= 2:
    print("\n" + "=" * 60)
    print("SECCION 6: ANALISIS DE CULTIVOS CLAVE")
    print("=" * 60)

    cultivos_exportacion = [
        "Aguacate",
        "Agave",
        "Palma africana o de aceite",
        "Maiz forrajero en verde",
        "Pastos y praderas",
        "Cana de azucar",
        "Nuez",
    ]
    cultivos_tradicionales = [
        "Maiz grano",
        "Frijol",
        "Sorgo grano",
        "Trigo grano",
        "Garbanzo grano",
        "Cartamo",
    ]

    cultivos_interes = cultivos_exportacion + cultivos_tradicionales

    df_filtrado = df_cultivos[df_cultivos["cultivo"].isin(cultivos_interes)].copy()

    pivot = df_filtrado.pivot_table(
        index="cultivo", columns="year", values="sembrada", aggfunc="sum"
    )
    if pivot.shape[1] >= 2:
        primer_year = pivot.columns[0]
        ultimo_year = pivot.columns[-1]
        pivot["cambio_ha"] = pivot[ultimo_year] - pivot[primer_year]
        pivot["cambio_pct"] = (
            (pivot[ultimo_year] - pivot[primer_year]) / pivot[primer_year]
        ) * 100

        print(f"\nCambio en ha sembradas por cultivo ({primer_year} vs {ultimo_year}):")
        print(
            pivot[["cambio_ha", "cambio_pct"]]
            .sort_values("cambio_ha", ascending=False)
            .round(0)
            .to_string()
        )

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        fig.suptitle(
            f"Transformacion Agricola en Mexico: Cultivos Seleccionados ({primer_year} vs {ultimo_year})",
            fontsize=13,
            fontweight="bold",
        )

        cultivos_ord = pivot.sort_values("cambio_ha")
        colores = [
            PALETTE["alerta"] if v > 0 else PALETTE["bosque"]
            for v in cultivos_ord["cambio_ha"].values
        ]

        ax = axes[0]
        bars = ax.barh(
            cultivos_ord.index,
            cultivos_ord["cambio_ha"] / 1000,
            color=colores,
            alpha=0.85,
            edgecolor="white",
        )
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title("Cambio absoluto en ha sembradas (miles)", fontsize=11)
        ax.set_xlabel("Miles de hectareas")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}k"))

        ax2 = axes[1]
        cultivos_pct = pivot.sort_values("cambio_pct")
        colores2 = [
            PALETTE["alerta"] if v > 0 else PALETTE["bosque"]
            for v in cultivos_pct["cambio_pct"].values
        ]
        ax2.barh(
            cultivos_pct.index,
            cultivos_pct["cambio_pct"],
            color=colores2,
            alpha=0.85,
            edgecolor="white",
        )
        ax2.axvline(0, color="black", linewidth=0.8)
        ax2.set_title("Cambio relativo (%)", fontsize=11)
        ax2.set_xlabel("Variacion porcentual")
        ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))

        plt.tight_layout()
        plt.savefig(
            os.path.join(OUTPUT_DIR, "05_cambio_cultivos_clave.png"),
            dpi=150,
            bbox_inches="tight",
        )
        plt.close()
        print("\nGuardado: 05_cambio_cultivos_clave.png")

    # Distribucion de ha sembradas por cultivo (top 20 en el primer ano disponible)
    primer = df_cultivos[df_cultivos["year"] == years_siap[0]].nlargest(20, "sembrada")
    ultimo = df_cultivos[df_cultivos["year"] == years_siap[-1]].nlargest(20, "sembrada")

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.suptitle(
        "Top 20 Cultivos por Superficie Sembrada", fontsize=13, fontweight="bold"
    )

    for ax_i, (df_plot, titulo) in enumerate(
        [(primer, str(years_siap[0])), (ultimo, str(years_siap[-1]))]
    ):
        ax = axes[ax_i]
        bars = ax.barh(
            df_plot["cultivo"].str[:30],
            df_plot["sembrada"] / 1000,
            color=PALETTE["neutro"],
            alpha=0.8,
            edgecolor="white",
        )
        ax.set_title(f"Year {titulo}", fontsize=11)
        ax.set_xlabel("Miles de hectareas")
        ax.invert_yaxis()
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}k"))

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "06_top20_cultivos.png"), dpi=150, bbox_inches="tight"
    )
    plt.close()
    print("Guardado: 06_top20_cultivos.png")


SECCION 6: ANALISIS DE CULTIVOS CLAVE

Cambio en ha sembradas por cultivo (2001 vs 2023):
year                        cambio_ha  cambio_pct
cultivo                                          
Pastos y praderas            872902.0        49.0
Aguacate                     170112.0       180.0
Agave                        149205.0       166.0
Nuez                         115017.0       222.0
Palma africana o de aceite    92952.0       495.0
Garbanzo grano              -121535.0       -62.0
Trigo grano                 -130859.0       -19.0
Sorgo grano                 -852857.0       -39.0
Frijol                      -861397.0       -44.0

Guardado: 05_cambio_cultivos_clave.png
Guardado: 06_top20_cultivos.png


---- SECCION 7: EVOLUCION POR ESTADO (SIAP) ----

In [14]:
if not df_estados.empty and len(years_siap) >= 2:
    print("\n" + "=" * 60)
    print("SECCION 7: CAMBIO AGRICOLA POR ESTADO")
    print("=" * 60)

    primer_year = years_siap[0]
    ultimo_year = years_siap[-1]
    e_primero = df_estados[df_estados["year"] == primer_year][
        ["estado", "sembrada"]
    ].rename(columns={"sembrada": f"sembrada_{primer_year}"})
    e_ultimo = df_estados[df_estados["year"] == ultimo_year][
        ["estado", "sembrada"]
    ].rename(columns={"sembrada": f"sembrada_{ultimo_year}"})
    estados_cambio = e_primero.merge(e_ultimo, on="estado")
    estados_cambio["cambio_ha"] = (
        estados_cambio[f"sembrada_{ultimo_year}"]
        - estados_cambio[f"sembrada_{primer_year}"]
    )
    estados_cambio["cambio_pct"] = (
        estados_cambio["cambio_ha"] / estados_cambio[f"sembrada_{primer_year}"] * 100
    )

    print(f"\nEstados con mayor GANANCIA agricola ({primer_year}-{ultimo_year}):")
    print(
        estados_cambio.nlargest(8, "cambio_ha")[
            ["estado", "cambio_ha", "cambio_pct"]
        ].to_string(index=False)
    )
    print(f"\nEstados con mayor PERDIDA agricola ({primer_year}-{ultimo_year}):")
    print(
        estados_cambio.nsmallest(8, "cambio_ha")[
            ["estado", "cambio_ha", "cambio_pct"]
        ].to_string(index=False)
    )

    fig, ax = plt.subplots(figsize=(12, 9))
    estados_ord = estados_cambio.sort_values("cambio_ha")
    colores = [
        PALETTE["alerta"] if v > 0 else PALETTE["bosque"]
        for v in estados_ord["cambio_ha"]
    ]
    ax.barh(
        estados_ord["estado"],
        estados_ord["cambio_ha"] / 1000,
        color=colores,
        alpha=0.85,
        edgecolor="white",
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(
        f"Cambio en Superficie Sembrada por Estado ({primer_year}-{ultimo_year})",
        fontsize=13,
        fontweight="bold",
    )
    ax.set_xlabel("Miles de hectareas")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}k"))
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "07_cambio_agricola_estados.png"),
        dpi=150,
        bbox_inches="tight",
    )
    plt.close()
    print("\nGuardado: 07_cambio_agricola_estados.png")


SECCION 7: CAMBIO AGRICOLA POR ESTADO

Estados con mayor GANANCIA agricola (2001-2023):
   estado  cambio_ha  cambio_pct
  Jalisco  228497.12   15.792840
   Oaxaca  177210.54   14.864586
 Campeche  135913.58   65.538261
Michoacán  119797.06   11.098458
 Guerrero  117290.99   14.555381
  Nayarit   42204.28   12.480435
 Veracruz   20910.54    1.367340
   Sonora   16938.64    3.103139

Estados con mayor PERDIDA agricola (2001-2023):
         estado  cambio_ha  cambio_pct
      Zacatecas -522199.39  -43.765166
        Sinaloa -305411.07  -23.095173
     Tamaulipas -301315.72  -19.635056
         México -172585.50  -18.903070
        Durango -137156.26  -20.271973
        Chiapas -126324.98   -8.516694
     Guanajuato -121968.24  -11.912683
San Luis Potosí -113151.12  -16.625077

Guardado: 07_cambio_agricola_estados.png


---- SECCION 8: CRUCE FORESTAL vs AGRICOLA ----

In [15]:
if not df_estados.empty and len(years_siap) >= 2:
    print("\n" + "=" * 60)
    print("SECCION 8: CRUCE PERDIDA FORESTAL vs CAMBIO AGRICOLA")
    print("=" * 60)

    # Normalizar nombres de estados para el merge
    gfw_estados["estado_norm"] = gfw_estados["subnational1"].str.strip()
    estados_cambio["estado_norm"] = estados_cambio["estado"].str.strip()

    cruce = gfw_estados[
        ["estado_norm", "perdida_total", "perdida_pct_extent", "extent_2000_ha"]
    ].merge(
        estados_cambio[["estado_norm", "cambio_ha", "cambio_pct"]],
        on="estado_norm",
        how="inner",
    )

    print(f"\nEstados en el cruce: {len(cruce)}")
    corr_abs = cruce["perdida_total"].corr(cruce["cambio_ha"])
    print(
        f"Correlacion de Pearson (perdida forestal vs cambio agricola): {corr_abs:.3f}"
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(
        "Perdida Forestal vs Cambio Agricola por Estado", fontsize=13, fontweight="bold"
    )

    ax = axes[0]
    colores_scatter = [
        PALETTE["alerta"] if v > 0 else PALETTE["bosque"] for v in cruce["cambio_ha"]
    ]
    scatter = ax.scatter(
        cruce["cambio_ha"] / 1000,
        cruce["perdida_total"] / 1000,
        c=colores_scatter,
        s=80,
        alpha=0.8,
        edgecolors="white",
        linewidth=0.5,
    )
    for _, row in cruce[cruce["perdida_total"] > 100000].iterrows():
        ax.annotate(
            row["estado_norm"],
            (row["cambio_ha"] / 1000, row["perdida_total"] / 1000),
            fontsize=7.5,
            xytext=(4, 4),
            textcoords="offset points",
            color=PALETTE["gris"],
        )
    ax.axvline(0, color="black", linewidth=0.6, linestyle="--")
    ax.set_xlabel("Cambio en ha agricolas (miles)")
    ax.set_ylabel("Ha bosque perdidas (miles)")
    ax.set_title("Perdida absoluta vs cambio agricola")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}k"))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}k"))

    ax2 = axes[1]
    scatter2 = ax2.scatter(
        cruce["cambio_pct"],
        cruce["perdida_pct_extent"],
        c=colores_scatter,
        s=80,
        alpha=0.8,
        edgecolors="white",
        linewidth=0.5,
    )
    for _, row in cruce[cruce["perdida_pct_extent"] > 10].iterrows():
        ax2.annotate(
            row["estado_norm"],
            (row["cambio_pct"], row["perdida_pct_extent"]),
            fontsize=7.5,
            xytext=(4, 4),
            textcoords="offset points",
            color=PALETTE["gris"],
        )
    ax2.axvline(0, color="black", linewidth=0.6, linestyle="--")
    ax2.set_xlabel("Cambio agricola (%)")
    ax2.set_ylabel("% de cobertura inicial perdida")
    ax2.set_title("Perdida relativa vs cambio agricola relativo")

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "08_scatter_forestal_agricola.png"),
        dpi=150,
        bbox_inches="tight",
    )
    plt.close()
    print("Guardado: 08_scatter_forestal_agricola.png")


SECCION 8: CRUCE PERDIDA FORESTAL vs CAMBIO AGRICOLA

Estados en el cruce: 31
Correlacion de Pearson (perdida forestal vs cambio agricola): 0.245
Guardado: 08_scatter_forestal_agricola.png


---- RESUMEN FINAL ----

In [17]:
print("\n" + "=" * 60)
print("RESUMEN EJECUTIVO DEL EDA")
print("=" * 60)

print(
    f"""
DATOS GFW (Cobertura Forestal)
  - Years disponibles:       2001-2023 ({len(years_gfw)} years)
  - Estados monitoreados:   {gfw_estados['subnational1'].nunique()}
  - Threshold utilizado:    {GFW_THRESHOLD}%
  - Perdida acumulada:      {gfw_estados['perdida_total'].sum():,.0f} ha
  - Estado mas afectado:    {gfw_estados.loc[gfw_estados['perdida_total'].idxmax(), 'subnational1']} ({gfw_estados['perdida_total'].max():,.0f} ha)
"""
)

if not df_estados.empty:
    anio_min, anio_max = years_siap[0], years_siap[-1]
    total_min = df_estados[df_estados["year"] == anio_min]["sembrada"].sum()
    total_max = df_estados[df_estados["year"] == anio_max]["sembrada"].sum()
    print(
        f"""DATOS SIAP (Agricultura)
  - Years disponibles:       {years_siap}
  - Cultivos unicos:        {df_cultivos['cultivo'].nunique()}
  - Total sembrado {anio_min}:   {total_min/1e6:.2f} millones ha
  - Total sembrado {anio_max}:   {total_max/1e6:.2f} millones ha
  - Cambio neto:            {(total_max-total_min)/1e6:.2f} millones ha ({(total_max-total_min)/total_min*100:.1f}%)
"""
    )

print(f"Graficas generadas en: {OUTPUT_DIR}/")
print("=" * 60)


RESUMEN EJECUTIVO DEL EDA

DATOS GFW (Cobertura Forestal)
  - Years disponibles:       2001-2023 (23 years)
  - Estados monitoreados:   32
  - Threshold utilizado:    30%
  - Perdida acumulada:      4,886,730 ha
  - Estado mas afectado:    Campeche (906,743 ha)

DATOS SIAP (Agricultura)
  - Years disponibles:       [np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
  - Cultivos unicos:        322
  - Total sembrado 2001:   21.59 millones ha
  - Total sembrado 2023:   20.00 millones ha
  - Cambio neto:            -1.59 millones ha (-7.4%)

Graficas generadas en: eda_output/
